In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

`GithubRepositoryDataReader` downloads the entire repository and goes over all the files in it. Because we specify `allowed_extensions={"md"}`, it only checks the markdown files.

We also pass a `filename_filter` so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's `lessons/` folder, so filtering on `/lessons/` keeps just those.

Each file has a `parse()` method that returns a dictionary with its `filename` and `content`:

### Q1. How many lesson pages

How many lesson pages are in the dataset?

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [6]:
documents[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

### Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and `filename` a keyword field. Then search with this query:

>How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

In [24]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

In [8]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [9]:
question = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    question,
    num_results=5
)


In [10]:
for r in results:
    print(r["filename"])
    print(r["content"][:100])
    print("---")

01-agentic-rag/lessons/14-agentic-loop.md
# The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3M
---
01-agentic-rag/lessons/15-frameworks.md
# ToyAIKit

Video: [Watch this lesson](https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_h
---
01-agentic-rag/lessons/13-function-calling.md
# Function Calling

Video: [Watch this lesson](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3M
---
01-agentic-rag/lessons/11-agents-intro.md
# Agents

Video: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZ
---
01-agentic-rag/lessons/16-other-frameworks.md
# Other Frameworks

Video: [Watch this lesson](https://www.youtube.com/watch?v=4yiCbKX9RhI&list=PL3M
---


### Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

`RAGBase` was written for the FAQ schema (section/question/answer), while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

In [19]:
question = "How does the agentic loop keep calling the model until it stops?"

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [20]:
# build context

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["filename"])
        lines.append(doc["content"])
        lines.append("")

    return "\n".join(lines).strip()

In [21]:
# build prompt

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

prompt = build_prompt(question, results)

In [25]:
# send prompt to LLM

from openai import OpenAI
openai_client = OpenAI()

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [51]:
print(response.usage.input_tokens)

7058


### Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding window - a window of `size` characters slides across the text in steps of `step` characters, and each window position becomes one chunk:



In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

With `size`=2000 and `step`=1000 (you can see the implementation here):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step` is smaller than `size`, consecutive chunks overlap by `size` - `step` (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the offset in the page) and `content` (the chunk text).

In [13]:
## How many chunks do you get?

len(chunks)

295

### Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field, `filename` as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

In [14]:
index.fit(chunks)

In [15]:
question = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    question,
    num_results=5
)


In [22]:
prompt = build_prompt(question, results)

In [26]:
# send prompt to LLM
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [27]:
print(response.usage.input_tokens)

2241


### Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give the LLM a `search` tool and let it decide when (and what) to search. We suggest toyaikit, the small agent library from the module, but you can use anything you like - the OpenAI Agents SDK, PydanticAI, LangChain, or a hand-written loop.


In [28]:
INSTRUCTIONS = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [30]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

In [31]:
# --- 1. Build your search index (reuse your existing documents) ---
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks) 

In [37]:
@tool
def search(query: str) -> str:
    """Search the course lesson content for entries matching the given query."""
    results = index.search(query, num_results=5)
    formatted = "\n\n".join(
        f"Source: {r['filename']}\n{r['content']}" for r in results
    )
    return formatted

llm = ChatOpenAI(model="gpt-4o-mini")

agent = create_agent(
    model=llm,
    tools=[search],
    system_prompt=INSTRUCTIONS
)

question = "How does the agentic loop work, and how is it different from plain RAG?"
response = agent.invoke({
    "messages": [{"role": "user", "content": question}]
})
print(response["messages"][-1].content)

The **agentic loop** is a framework used to create dynamic agent-based workflows in AI applications. It allows an AI model to interactively engage in a back-and-forth conversation with users or other systems, responding to input, executing tool calls, and maintaining a history of messages. The key components of an agentic loop include:

1. **Loop Execution**: The loop continually processes input until a final output is produced. It's designed to allow the model to make iterative decisions based on previous messages.
2. **State Management**: It keeps track of the conversation history, which is essential for understanding context and for the model to make informed responses in subsequent turns.
3. **Flexibility**: The framework allows the agent to decide what actions to take next based on the situation and goals, in contrast to fixed workflows that follow predetermined paths.

In comparison, **RAG (Retrieval Augmented Generation)** is a technique that enhances the capabilities of languag

In [39]:
tool_messages = [msg for msg in response["messages"] if msg.type == "tool"]
print(f"Search was called {len(tool_messages)} time(s)")

Search was called 2 time(s)
